# Feature Extraction Fine-tuning — Llama-3.2-1B-Instruct

**Strategy:** Feature-Based (Feature Extraction) — giữ nguyên toàn bộ trọng số
của model nền (frozen), chỉ huấn luyện một lớp phân loại (classifier head) ở cuối.

**Model:** `meta-llama/Llama-3.2-1B-Instruct` (1B params)

**Dataset:** `code_x_glue_cc_defect_detection`

**Ưu điểm:** Nhanh, ít tốn tài nguyên, tận dụng kiến thức sâu của pretrained model.

**Nhược điểm:** Có thể kém hiệu quả hơn full fine-tuning vì model không được
tối ưu trực tiếp cho task.

**Cách tiếp cận:** Dùng hidden states từ token cuối cùng của LLM làm features,
đưa qua một classification head (Linear layer) để phân loại binary.

## 1. Setup

In [ ]:
!pip install -q transformers datasets accelerate bitsandbytes
!pip install -q scikit-learn matplotlib seaborn huggingface_hub

from huggingface_hub import login
login()  # Llama 3.2 is a gated model — paste your HF token when prompted

In [ ]:
import os
import sys
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm import tqdm
import numpy as np

# On Colab, clone the repo first:
# !git clone -b trung https://github.com/MinhTuan2405/IE105.21_SBugDwLLM.git
# %cd IE105.21_SBugDwLLM/finetuning/llama32

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from utils.data_loader import load_defect_dataset, build_chat_messages, SYSTEM_PROMPT
from utils.evaluation import evaluate_predictions

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
OUTPUT_DIR = "./llama32-feature-extraction"
RESULTS_DIR = "../../docs/llama32_docs"

MAX_SEQ_LENGTH = 1024
CLASSIFIER_EPOCHS = 10
CLASSIFIER_LR = 1e-3
BATCH_SIZE = 16

## 3. Load Model (Frozen) and Tokenizer

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    output_hidden_states=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Freeze all model parameters
for param in model.parameters():
    param.requires_grad = False

model.eval()
print(f"Model loaded (FROZEN). Memory: {model.get_memory_footprint() / 1e9:.2f} GB")
print(f"Hidden size: {model.config.hidden_size}")

## 4. Extract Features

Chạy toàn bộ data qua frozen LLM, lấy hidden state của token cuối cùng
làm feature vector cho mỗi sample.

In [ ]:
def extract_features(dataset, model, tokenizer, max_length=1024, desc="Extracting"):
    """Extract last-token hidden states from frozen LLM."""
    device = next(model.parameters()).device
    features = []
    labels = []

    for sample in tqdm(dataset, desc=desc):
        messages = build_chat_messages(sample["func"])
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        inputs = tokenizer(
            prompt, return_tensors="pt",
            truncation=True, max_length=max_length,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        # Take the last token's hidden state from the last layer
        last_hidden = outputs.hidden_states[-1]  # (1, seq_len, hidden_size)
        seq_lengths = inputs["attention_mask"].sum(dim=1) - 1
        last_token_hidden = last_hidden[0, seq_lengths[0], :]  # (hidden_size,)

        features.append(last_token_hidden.cpu().float())
        labels.append(sample["target"])

    return torch.stack(features), torch.tensor(labels)

In [ ]:
train_data, val_data, test_data = load_defect_dataset()

print("Extracting training features...")
X_train, y_train = extract_features(train_data, model, tokenizer, MAX_SEQ_LENGTH, "Train")
print(f"Train features shape: {X_train.shape}")

print("\nExtracting validation features...")
X_val, y_val = extract_features(val_data, model, tokenizer, MAX_SEQ_LENGTH, "Val")

print("\nExtracting test features...")
X_test, y_test = extract_features(test_data, model, tokenizer, MAX_SEQ_LENGTH, "Test")
print(f"\nFeature dim: {X_train.shape[1]}")

## 5. Train Classification Head

In [ ]:
class ClassifierHead(nn.Module):
    def __init__(self, hidden_size, num_classes=2):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(x)


class FeatureDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


hidden_size = X_train.shape[1]
classifier = ClassifierHead(hidden_size).cuda()

train_loader = DataLoader(FeatureDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(FeatureDataset(X_val, y_val), batch_size=BATCH_SIZE)

optimizer = torch.optim.Adam(classifier.parameters(), lr=CLASSIFIER_LR)
criterion = nn.CrossEntropyLoss()

print(f"Classifier params: {sum(p.numel() for p in classifier.parameters()):,}")

In [ ]:
best_val_acc = 0

for epoch in range(CLASSIFIER_EPOCHS):
    # Train
    classifier.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.cuda(), y_batch.cuda()
        optimizer.zero_grad()
        logits = classifier(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Validate
    classifier.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.cuda(), y_batch.cuda()
            preds = classifier(X_batch).argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    val_acc = correct / total
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{CLASSIFIER_EPOCHS} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        torch.save(classifier.state_dict(), os.path.join(OUTPUT_DIR, "best_classifier.pt"))
        print(f"  -> Saved best model (val_acc={val_acc:.4f})")

print(f"\nBest validation accuracy: {best_val_acc:.4f}")

## 6. Evaluation on Test Set

In [ ]:
# Load best classifier
classifier.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "best_classifier.pt")))
classifier.eval()

test_loader = DataLoader(FeatureDataset(X_test, y_test), batch_size=BATCH_SIZE)

y_true_list, y_pred_list = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.cuda()
        preds = classifier(X_batch).argmax(dim=1)
        y_true_list.extend(y_batch.tolist())
        y_pred_list.extend(preds.cpu().tolist())

os.makedirs(RESULTS_DIR, exist_ok=True)

metrics = evaluate_predictions(
    y_true_list, y_pred_list,
    model_name="llama32",
    strategy="feature_extraction",
    save_dir=RESULTS_DIR,
)

## 7. Error Analysis

In [ ]:
errors = [
    {"index": i, "true": yt, "pred": yp, "code": test_data[i]["func"][:200]}
    for i, (yt, yp) in enumerate(zip(y_true_list, y_pred_list))
    if yt != yp
]

false_positives = [e for e in errors if e["true"] == 0 and e["pred"] == 1]
false_negatives = [e for e in errors if e["true"] == 1 and e["pred"] == 0]

print(f"Total errors: {len(errors)}")
print(f"False Positives: {len(false_positives)}")
print(f"False Negatives: {len(false_negatives)}")

print("\n--- Sample False Negatives ---")
for e in false_negatives[:3]:
    print(f"  idx={e['index']}: {e['code'][:100]}...")

print("\n--- Sample False Positives ---")
for e in false_positives[:3]:
    print(f"  idx={e['index']}: {e['code'][:100]}...")